## 4. Volatilité, liquidité et ampleur des écarts

On se demande ici si les primes et décotes importants, c'est-à-dire les écarts importants entre NAV et prix de l'ETF sont liés aux conditions de marché (volatilité et liquidité).

Le VIX sert à distinguer les régimes de volatilité faible et forte. La règle appliquée est :

$$R_{\mathrm{vol},t} = \begin{cases} \text{Faible} & \text{si } VIX_t < 15 \\ \text{Forte} & \text{si } VIX_t \geq 30 \\ \text{Intermédiaire} & \text{sinon} \end{cases}$$

Pour la liquidité, les quartiles sont calculés séparément pour chaque ETF :

$$R_{\mathrm{liq},t} = \begin{cases} \text{Faible} & \text{si } Volume_t \leq Q_{25} \\ \text{Forte} & \text{si } Volume_t \geq Q_{75} \\ \text{Intermédiaire} & \text{sinon} \end{cases}$$

La cellule compare ensuite l'écart absolu moyen entre les régimes faible et fort, sans prétendre établir une causalité.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

RACINE = Path.cwd()
if not (RACINE / "annexes").is_dir():
    RACINE = RACINE.parent
if str(RACINE) not in sys.path:
    sys.path.insert(0, str(RACINE))

from annexes.a_donnees import charger_nav, charger_prix, charger_vix
from annexes.b_analyses import construire_donnees, ajouter_regimes, comparer_regimes, analyser_correction
from annexes.c_graphiques import tracer_comparaison_regimes, tracer_corrections
import matplotlib.pyplot as plt


In [ ]:
nav = charger_nav()
prix = charger_prix()
donnees = construire_donnees(nav, prix)
date_debut = donnees["Date"].min().strftime("%Y-%m-%d")
date_fin = donnees["Date"].max().strftime("%Y-%m-%d")
vix = charger_vix(date_debut, date_fin)
donnees = ajouter_regimes(donnees, vix)
comparaison_volatilite = comparer_regimes(donnees, "Regime_volatilite")
comparaison_liquidite = comparer_regimes(donnees, "Regime_liquidite")
display(comparaison_volatilite.round(4))
display(comparaison_liquidite.round(4))

fig = tracer_comparaison_regimes(donnees)
display(fig)
plt.close(fig)

### Résultats de l'analyse des conditions de marché

En forte volatilité, l'écart absolu moyen augmente pour les quatre catégories : **0,0202 % → 0,0671 %** pour les actions américaines, **0,3752 % → 0,8283 %** pour les actions émergentes, **0,1314 % → 0,3412 %** pour l'investment grade et **0,3323 % → 0,5096 %** pour le high yield. L'effet est particulièrement marqué pour SPEM.

L'effet du volume est hétérogène : l'écart augmente lorsque le volume est fort pour les actions américaines et émergentes, mais diminue pour les obligations (**0,4625 % → 0,1722 %** pour le high yield ; **0,2503 % → 0,0618 %** pour l'investment grade). Le volume est donc un proxy imparfait de la liquidité.

## 5. Correction le jour suivant

On se demande si les écarts se rétablissent le jour d'après, quand les opérateurs de marché arbitrent la différence entre la valeur de la NAV et des ETF.

On retient les écarts absolus d'au moins 0,25 %, puis on compare leur valeur avec celle de la séance suivante. La variation de l'écart est définie par :

$$\Delta_t = \mathrm{Écart\ absolu}_{t+1} - \mathrm{Écart\ absolu}_t$$

Une correction est comptée lorsque :

$$\mathrm{Correction}_t = \mathbb{1}(\Delta_t < 0)$$

Le taux de correction est ensuite :

$$\mathrm{Taux\ de\ correction} = \frac{\text{nombre de corrections}}{\text{nombre d'écarts importants observés}} \times 100$$

Dans le nuage de points, les observations sous la diagonale correspondent à une correction et celles au-dessus à une aggravation de l'écart enregistré le jour précédent.

In [ ]:
donnees_ecarts_importants, vitesse_correction = analyser_correction(donnees)
display(vitesse_correction.round(4))

fig = tracer_corrections(donnees_ecarts_importants)
display(fig)
plt.close(fig)

### Résultats de l'étude des corrections

Le taux de correction des écarts importants est de **68,67 % pour SPEM** (variation moyenne : −0,1663 point de pourcentage), **67,28 % pour SPHY** (−0,1050), **65,22 % pour SPIB** (−0,0526) et **100 % pour SPY** (−0,3203).

On observe donc que dans la majorité des cas les arbitrages mènent les écarts à se résorber.

Le résultat de SPY doit être interprété avec prudence car il ne repose que sur **11 dates**. Dans l'ensemble, les écarts importants ont tendance à diminuer dès la séance suivante, sans disparaître systématiquement.

# Conclusion et limites

## Conclusion

L'ETF et le portefeuille sous-jacent peuvent représenter la même exposition tout en ayant des prix différents, car le prix de l'ETF est déterminé en continu par l'offre et la demande alors que la NAV est calculée à partir de la valorisation des actifs du portefeuille, souvent à un autre moment et avec d'autres conventions. Les résultats montrent que les écarts sont faibles pour SPY, plus importants pour SPEM et SPHY, qu'ils s'amplifient généralement lorsque la volatilité est forte et qu'ils se corrigent souvent dès le lendemain.

## Limites du projet

- La NAV et le prix de clôture peuvent correspondre à des moments différents de la même journée.
- Le VIX est un indicateur de volatilité du S&P 500, pas une mesure spécifique de chacun des ETF.
- Les seuils `0,25 %`, `0,5 %`, `1 %`, `VIX < 15` et `VIX ≥ 30` sont des choix d'analyse ; ils ne constituent pas des seuils universels.
- Le volume est un proxy imparfait de la liquidité.
- Le projet ne rend pas compte de causalité mais de tendances.